# MSWC + Common Voice metadata analysis

This notebook audits the two metadata sources separately, their raw filename-based join, and the final processed word-contrastive corpus. It is read-only: no dataset file is modified.

- **MSWC** supplies the aligned one-second word clips, word labels, and canonical speaker IDs.
- **Common Voice 21** supplies accent metadata and source transcripts.
- The join key is the original Common Voice clip filename reconstructed from the MSWC `LINK` field.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

ROOT = Path.cwd().resolve()
if not (ROOT / "pixi.toml").exists():
    raise RuntimeError("Run this notebook with the repository root as the working directory.")

MSWC_DIR = ROOT / "data/raw/mswc_common_voice_en/mswc"
CV_TSV = ROOT / "data/raw/mswc_common_voice_en/common_voice_21/validated.tsv"
PROCESSED_DIR = ROOT / "data/processed/mswc_common_voice_words/en"
CORPUS_PARQUET = PROCESSED_DIR / "corpus.parquet"

for path in (MSWC_DIR, CV_TSV, CORPUS_PARQUET):
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

In [ ]:
def overview(frame: pd.DataFrame, name: str) -> None:
    memory_mb = frame.memory_usage(deep=True).sum() / 1024**2
    print(f"{name}: {len(frame):,} rows × {len(frame.columns)} columns; {memory_mb:,.1f} MiB in memory")
    print("Columns:", list(frame.columns))
    display(frame.head())
    display(pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "non_null": frame.notna().sum(),
        "missing": frame.isna().sum(),
        "missing_percent": frame.isna().mean().mul(100),
        "unique": frame.nunique(dropna=True),
    }))

def source_filename(link: str) -> str:
    name = Path(str(link)).name
    stem = re.sub(r"__\d+$", "", Path(name).stem)
    return f"{stem}.mp3"

def normalized_word(value: object) -> str:
    return re.sub(r"[^a-z']+", "", str(value).strip().casefold())

## 1. MSWC metadata before the join

MSWC provides three CSV files. We first inspect each original split, then concatenate them exactly as the preparation pipeline does.

In [ ]:
mswc_frames = []
mswc_file_stats = []
for original_split in ("train", "dev", "test"):
    csv_path = MSWC_DIR / f"en_{original_split}.csv"
    frame = pd.read_csv(csv_path)
    frame["mswc_original_split"] = original_split
    mswc_frames.append(frame)
    mswc_file_stats.append({
        "split": original_split,
        "rows": len(frame),
        "columns": len(frame.columns),
        "valid_rows": frame["VALID"].astype(str).str.casefold().eq("true").sum(),
        "speakers": frame["SPEAKER"].nunique(),
        "words": frame["WORD"].nunique(),
    })

display(pd.DataFrame(mswc_file_stats))
overview(mswc_frames[0], "MSWC train metadata (unfiltered)")

In [ ]:
mswc_raw = pd.concat(mswc_frames, ignore_index=True)
mswc_valid = mswc_raw.loc[
    mswc_raw["VALID"].astype(str).str.casefold().eq("true")
].copy()
mswc_valid["source_filename"] = mswc_valid["LINK"].map(source_filename)
mswc_valid["normalized_word"] = mswc_valid["WORD"].map(normalized_word)
mswc_valid = mswc_valid.loc[mswc_valid["normalized_word"].ne("")].copy()

print(f"All MSWC rows: {len(mswc_raw):,}")
print(f"Valid rows with a non-empty normalized word: {len(mswc_valid):,}")
print(f"Distinct clips: {mswc_valid['source_filename'].nunique():,}")
print(f"Distinct aligned occurrences (LINK): {mswc_valid['LINK'].nunique():,}")
print(f"Distinct speakers: {mswc_valid['SPEAKER'].nunique():,}")
print(f"Distinct normalized words: {mswc_valid['normalized_word'].nunique():,}")
print(f"Duplicated LINK values: {mswc_valid['LINK'].duplicated().sum():,}")
display(mswc_valid["normalized_word"].value_counts().describe(percentiles=[.01, .1, .25, .5, .75, .9, .99]).to_frame("occurrences_per_word"))
display(mswc_valid["normalized_word"].value_counts().head(20).to_frame("occurrences"))

## 2. Common Voice metadata before the join

This is the pinned Common Voice 21 `validated.tsv`. It supplies the accent field, client identifier, and original sentence.

In [ ]:
cv_raw = pd.read_csv(CV_TSV, sep="\t", keep_default_na=False)
overview(cv_raw, "Common Voice 21 validated metadata")

accent_column = "accent" if "accent" in cv_raw.columns else "accents"
cv = cv_raw.rename(columns={accent_column: "accent"}).copy()
cv["source_filename"] = cv["path"].map(lambda value: Path(str(value)).name)
cv["accent"] = cv["accent"].astype(str).str.strip().str.casefold()

print(f"Rows: {len(cv):,}")
print(f"Distinct clip filenames: {cv['source_filename'].nunique():,}")
print(f"Duplicate clip filenames: {cv['source_filename'].duplicated().sum():,}")
print(f"Distinct client IDs: {cv['client_id'].nunique():,}")
print(f"Rows with a non-empty accent: {cv['accent'].ne('').sum():,} ({cv['accent'].ne('').mean():.2%})")
print(f"Distinct non-empty accent labels: {cv.loc[cv.accent.ne(''), 'accent'].nunique():,}")
display(cv.loc[cv["accent"].ne("")].groupby("accent").agg(
    clips=("source_filename", "size"),
    speakers=("client_id", "nunique"),
).sort_values("clips", ascending=False))

## 3. Raw join audit

The production join is many MSWC word segments to one Common Voice source clip. `_merge` distinguishes unmatched filenames from matched clips. A matched row is not necessarily retained: it must also have a non-empty accent and belong to an eligible word.

In [ ]:
cv_join_columns = cv[["source_filename", "client_id", "accent", "sentence"]].copy()
joined = mswc_valid.merge(
    cv_join_columns, on="source_filename", how="left",
    validate="many_to_one", indicator=True,
)
overview(joined, "MSWC–Common Voice raw join")

joined["has_accent"] = joined["accent"].fillna("").astype(str).str.strip().ne("")
joined["speaker_hash_matches"] = (
    joined["SPEAKER"].fillna("").astype(str)
    == joined["client_id"].fillna("").astype(str)
)
matched = joined["_merge"].eq("both")
join_summary = pd.Series({
    "MSWC valid word occurrences": len(joined),
    "matched occurrences": int(matched.sum()),
    "unmatched occurrences": int((~matched).sum()),
    "occurrence join rate": float(matched.mean()),
    "matched occurrences with accent": int((matched & joined['has_accent']).sum()),
    "accent availability among matched": float(joined.loc[matched, 'has_accent'].mean()),
    "speaker hash matches among matched": int(joined.loc[matched, 'speaker_hash_matches'].sum()),
    "speaker hash match rate": float(joined.loc[matched, 'speaker_hash_matches'].mean()),
})
display(join_summary.to_frame("value"))
display(joined.loc[~matched, ["LINK", "source_filename", "WORD", "SPEAKER"]].head(20))

## 4. Final processed corpus after the join and vocabulary filtering

The final corpus contains only words satisfying the training-set coverage rule: at least six accent groups with at least three distinct speakers per accent. The train/dev split is speaker-disjoint.

In [ ]:
corpus = pd.read_parquet(CORPUS_PARQUET)
overview(corpus, "Final processed word-contrastive corpus")

basic_stats = corpus.groupby("split").agg(
    examples=("audio_path", "size"),
    hours=("duration_s", lambda values: values.sum() / 3600),
    speakers=("speaker_id", "nunique"),
    accents=("accent", "nunique"),
    words=("normalized_word", "nunique"),
    source_utterances=("utterance_id", "nunique"),
).reset_index()
display(basic_stats)
display(pd.Series({
    "examples": len(corpus),
    "hours": corpus['duration_s'].sum() / 3600,
    "speakers": corpus['speaker_id'].nunique(),
    "accents": corpus['accent'].nunique(),
    "words": corpus['normalized_word'].nunique(),
    "duplicate audio paths": corpus['audio_path'].duplicated().sum(),
    "missing audio paths in metadata": corpus['audio_path'].isna().sum(),
}).to_frame("value"))

In [ ]:
train_speakers = set(corpus.loc[corpus.split.eq("train"), "speaker_id"])
dev_speakers = set(corpus.loc[corpus.split.eq("dev"), "speaker_id"])
train_words = set(corpus.loc[corpus.split.eq("train"), "normalized_word"])
dev_words = set(corpus.loc[corpus.split.eq("dev"), "normalized_word"])

display(pd.Series({
    "speaker overlap train/dev": len(train_speakers & dev_speakers),
    "words shared by train/dev": len(train_words & dev_words),
    "train-only words": len(train_words - dev_words),
    "dev-only words": len(dev_words - train_words),
}).to_frame("value"))

accent_stats = corpus.groupby(["split", "accent"]).agg(
    examples=("audio_path", "size"),
    hours=("duration_s", lambda values: values.sum() / 3600),
    speakers=("speaker_id", "nunique"),
    words=("normalized_word", "nunique"),
).reset_index()
display(accent_stats.sort_values(["split", "examples"], ascending=[True, False]))

In [ ]:
train = corpus.loc[corpus.split.eq("train")].copy()
word_stats = train.groupby("normalized_word").agg(
    examples=("audio_path", "size"),
    speakers=("speaker_id", "nunique"),
    accents=("accent", "nunique"),
).sort_values("examples", ascending=False)

print("Distribution across retained training words")
display(word_stats.describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]))
print("Most frequent words")
display(word_stats.head(25))
print("Least frequent retained words")
display(word_stats.tail(25).sort_values("examples"))

word_accent = train.groupby(["normalized_word", "accent"]).agg(
    examples=("audio_path", "size"),
    speakers=("speaker_id", "nunique"),
).reset_index()
display(word_accent[["examples", "speakers"]].describe(
    percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]
))

## Interpretation checklist

Before training, verify the following from the outputs above:

1. The filename join rate and accent availability are reported separately.
2. Speaker overlap between train and dev is exactly zero.
3. Accent labels are meaningful and not fragmented aliases requiring normalization.
4. No small set of accents, words, or speakers dominates the corpus.
5. The 27.1-hour controlled subset should be sampled by word, accent, and speaker—not uniformly by row.
6. The full 741-hour corpus should be treated as a scaling experiment, not as the size-matched comparison with L2-ARCTIC.